# PySpark `partitionBy()` Demo — Week 9

**What this notebook covers:**
- Creating a `SparkSession` on a YARN cluster with Hive support
- Reading a large CSV file (~2.8 million rows) into a DataFrame
- Writing data to disk **partitioned by column(s)** using `.partitionBy()`
- Comparing query performance/behavior on partitioned vs non-partitioned data
- Partitioning by a **single column** (`order_status`) and by **multiple columns** (`customer_state`, `customer_city`)

**Key idea:** `partitionBy()` physically splits output data into sub-folders based on column values 
(e.g. `order_status=CLOSED/`, `order_status=COMPLETE/`, ...). When you later query with a filter on that 
column, Spark can use **partition pruning** — it skips reading folders that don't match the filter, 
instead of scanning the entire dataset. This can massively speed up queries on large datasets.

In [1]:
# Import SparkSession - the entry point for all Spark functionality (DataFrame/SQL API)
from pyspark.sql import SparkSession
import getpass  # used to fetch the current OS username dynamically

# Get current logged-in username (used to build a user-specific warehouse path)
username = getpass.getuser()

spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config('spark.shuffle.useOldFetchProtocol', 'true'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

# config('spark.ui.port','0')                 -> lets Spark pick a random free port for the Spark UI (avoids port clashes on shared clusters)
# config('spark.shuffle.useOldFetchProtocol')  -> compatibility setting for shuffle data fetching on some YARN/Hadoop versions
# config("spark.sql.warehouse.dir", ...)       -> sets the default location for Hive/Spark SQL managed tables, scoped to this user
# enableHiveSupport()                          -> allows use of Hive metastore, HQL syntax, and Hive tables
# master('yarn')                               -> run Spark on a YARN-managed cluster (as opposed to 'local[*]')
# getOrCreate()                                -> reuse an existing SparkSession if one exists, otherwise create a new one

## Step 1: Define schema and read the raw orders data
Instead of relying on `inferSchema` (slow, requires an extra pass over the data), we explicitly 
define the schema as a DDL-style string. This is faster and avoids incorrect type inference.

In [2]:
# Define schema explicitly as a DDL string: "column_name data_type, column_name data_type, ..."
# This avoids the overhead/inaccuracy of inferSchema=True on a large file
orders_schema = "order_id long , order_date string, cust_id long,order_status string"

In [3]:
# Read the raw (non-partitioned) orders CSV file from HDFS using the explicit schema above
orders_df = spark.read \
.format("csv") \
.schema(orders_schema) \
.load("/public/trendytech/orders/orders_1gb.csv")

In [4]:
# Preview the first 20 rows to sanity-check the schema and data loaded correctly
orders_df.show()

+--------+--------------------+-------+---------------+
|order_id|          order_date|cust_id|   order_status|
+--------+--------------------+-------+---------------+
|       1|2013-07-25 00:00:...|  11599|         CLOSED|
|       2|2013-07-25 00:00:...|    256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|  12111|       COMPLETE|
|       4|2013-07-25 00:00:...|   8827|         CLOSED|
|       5|2013-07-25 00:00:...|  11318|       COMPLETE|
|       6|2013-07-25 00:00:...|   7130|       COMPLETE|
|       7|2013-07-25 00:00:...|   4530|       COMPLETE|
|       8|2013-07-25 00:00:...|   2911|     PROCESSING|
|       9|2013-07-25 00:00:...|   5657|PENDING_PAYMENT|
|      10|2013-07-25 00:00:...|   5648|PENDING_PAYMENT|
|      11|2013-07-25 00:00:...|    918| PAYMENT_REVIEW|
|      12|2013-07-25 00:00:...|   1837|         CLOSED|
|      13|2013-07-25 00:00:...|   9149|PENDING_PAYMENT|
|      14|2013-07-25 00:00:...|   9842|     PROCESSING|
|      15|2013-07-25 00:00:...|   2568|       CO

#### Note : Please replace external id "itv006277" with your id number

## Step 2: Read data from a previously written (non-partitioned) location
This re-reads `orders_df` from a path (`sparkwriterdemo1`) that was written earlier (in a prior demo), 
so we have a **baseline, non-partitioned** dataset to compare against later.

In [5]:
# Re-read orders data, this time from a path written out earlier (acts as our baseline/non-partitioned dataset)
orders_df = spark.read \
.format("csv") \
.schema(orders_schema) \
.load("/user/itv006277/sparkwriterdemo1")

In [6]:
# Register the DataFrame as a temporary SQL view called "orders" so we can run Spark SQL queries on it
orders_df.createOrReplaceTempView("orders")

### Baseline query (no partitioning yet)
This query has to scan the **entire dataset** because the data on disk is not partitioned by `order_status`.
We'll compare this behavior with the partitioned version later.

In [7]:
# Count CLOSED orders. Since data is NOT partitioned on disk yet, Spark must scan all files/rows (full scan).
spark.sql("select count(*) from orders where order_status = 'CLOSED'")

count(1)
2833500


## Step 3: Write data partitioned by a single column — `order_status`
`partitionBy("order_status")` tells Spark to create one **sub-folder per distinct value** of `order_status` 
under the output path, e.g.:
```
partition_demo_output1/
  order_status=CLOSED/
  order_status=COMPLETE/
  order_status=PENDING_PAYMENT/
  ...
```
Each folder contains only the rows matching that value. Future queries that filter on `order_status` 
can then skip irrelevant folders entirely (partition pruning).

In [8]:
# Write orders_df to disk, partitioned by order_status (creates one sub-folder per distinct status value)
orders_df.write \
.format("csv")\
.mode("overwrite") \
.partitionBy("order_status") \
.option("path", "/user/itv006277/partition_demo_output1") \
.save()

# .mode("overwrite")        -> replace existing data at this path if it already exists
# .partitionBy("order_status") -> physically split output into folders like order_status=CLOSED/, order_status=COMPLETE/, etc.
# .option("path", ...)      -> destination directory on HDFS
# Note: the order_status column itself is DROPPED from the actual data files and instead encoded
# in the folder path/name (this is standard Hive-style partitioning).

In [9]:
# Number of IN-MEMORY RDD partitions of orders_df (this is about parallelism/tasks in memory,
# NOT the same concept as the on-disk partitionBy folders written above!)
orders_df.rdd.getNumPartitions()

9

In [10]:
# Count of distinct order_status values -> this tells us how many partition folders were created
# by the write above (one folder per distinct order_status value)
spark.sql("select distinct(order_status) from orders").count()

9

## Step 4: Read back the partitioned data and compare query behavior
Now we read from `partition_demo_output1` (the partitioned output). Spark automatically detects 
the `order_status=...` folder structure and treats `order_status` as a column again 
(this is called **partition discovery**).

In [11]:
# Read back the data we just wrote, now partitioned by order_status on disk
orders_df = spark.read \
.format("csv") \
.schema(orders_schema) \
.load("/user/itv006277/partition_demo_output1")

In [12]:
# Re-register as a temp view so SQL queries below read from the PARTITIONED dataset
orders_df.createOrReplaceTempView("orders")

### Query on the partitioned data (filter matches the partition column)
Because `order_status` is the partition column, Spark can jump straight to the 
`order_status=CLOSED/` folder instead of scanning every file — this is **partition pruning** 
in action. Same result as the baseline query above, but far less I/O on a large dataset.

In [13]:
# Same query as before, but now Spark can prune directly to the order_status=CLOSED/ folder
spark.sql("select count(*) from orders where order_status = 'CLOSED'")

count(1)
2833500


In [14]:
# Another partition-pruned query: only reads the order_status=PENDING_PAYMENT/ folder
spark.sql("select count(*) from orders where order_status = 'PENDING_PAYMENT'")

count(1)
5636250


### Query that does NOT filter on the partition column
This filters only on `cust_id`, which is NOT the partition column. Spark must scan across 
**all** `order_status=...` folders to find matching rows — no pruning benefit here.

In [15]:
# Filtering only on cust_id (not the partition column) -> Spark must scan ALL partition folders
spark.sql("select count(*) from orders where cust_id = 8827")

count(1)
2250


### Combined filter: partition column + non-partition column
Filtering on both `order_status` (partition column) AND `cust_id` (regular column) still lets 
Spark prune down to just the `order_status=CLOSED/` folder first, then filter `cust_id` within 
that smaller set of files.

In [16]:
# order_status filter enables partition pruning to the CLOSED folder; cust_id filter applied within it
spark.sql("select count(*) from orders where order_status = 'CLOSED' and cust_id = 8827")

count(1)
375


In [17]:
# Same filters, just written in the opposite order in the WHERE clause.
# Spark's optimizer (Catalyst) reorders/analyzes predicates itself, so the column order in the
# WHERE clause doesn't matter -> pruning still happens, and the result matches the cell above (375)
spark.sql("select count(*) from orders where cust_id = 8827 and order_status = 'CLOSED'")

count(1)
375


## Step 5: Partitioning by MULTIPLE columns — `customer_state`, `customer_city`
Now we switch to the `customers` dataset and demonstrate **multi-level (nested) partitioning**. 
`partitionBy("customer_state", "customer_city")` creates a nested folder hierarchy:
```
partition_demo_output2/
  customer_state=PR/
    customer_city=Caguas/
    customer_city=San Juan/
    ...
  customer_state=CA/
    customer_city=...
```
The order of columns in `partitionBy()` determines the folder nesting order (state, then city).

In [18]:
# Read the raw customers CSV, letting Spark infer the schema (fine for smaller reference-type data)
customers_df = spark.read \
.format("csv") \
.option("inferSchema", True) \
.load("/public/trendytech/retail_db/customers/part-00000")

In [19]:
# The source CSV has no header, so rename the default (_c0, _c1, ...) columns to meaningful names
customers_final_df = customers_df.toDF("customer_id", "customer_fname", "customer_lname", "customer_email", "customer_password", "customer_street", "customer_city", "customer_state", "customer_zipcode")

In [20]:
# Write customers data as Parquet, partitioned by BOTH customer_state and customer_city
# -> creates nested folders: customer_state=XX/customer_city=YY/
customers_final_df.write \
.format("parquet")\
.mode("overwrite") \
.partitionBy("customer_state","customer_city") \
.option("path", "/user/itv006277/partition_demo_output2") \
.save()

# Note: using Parquet here (columnar, compressed) instead of CSV — a common pairing with
# partitioning for analytical workloads, since Parquet also stores column stats per file.

In [21]:
# Read back the partitioned Parquet data. Parquet stores its own schema, so no .schema()/.option("inferSchema") needed
customers_df = spark.read \
.format("parquet") \
.load("/user/itv006277/partition_demo_output2")

In [22]:
# Preview the data. Notice customer_state and customer_city appear as normal columns again
# (Spark reconstructs them from the folder names via partition discovery), even though they were
# not physically stored inside the Parquet files themselves.
customers_df.show()

+-----------+--------------+--------------+--------------+-----------------+--------------------+----------------+--------------+-------------+
|customer_id|customer_fname|customer_lname|customer_email|customer_password|     customer_street|customer_zipcode|customer_state|customer_city|
+-----------+--------------+--------------+--------------+-----------------+--------------------+----------------+--------------+-------------+
|          3|           Ann|         Smith|     XXXXXXXXX|        XXXXXXXXX|3422 Blue Pioneer...|             725|            PR|       Caguas|
|          5|        Robert|        Hudson|     XXXXXXXXX|        XXXXXXXXX|10 Crystal River ...|             725|            PR|       Caguas|
|          7|       Melissa|        Wilcox|     XXXXXXXXX|        XXXXXXXXX|9453 High Concession|             725|            PR|       Caguas|
|          9|          Mary|         Perez|     XXXXXXXXX|        XXXXXXXXX| 3616 Quaking Street|             725|            PR|       

In [23]:
# In-memory RDD partition count of customers_final_df (the version BEFORE the partitioned write) —
# again, this is unrelated to the on-disk partitionBy folder structure
customers_final_df.rdd.getNumPartitions()

1

In [24]:
# Register the partitioned customers data as a temp view for SQL querying
customers_df.createOrReplaceTempView("customers")

### Query using both partition columns (state + city)
Filtering on `customer_state` and `customer_id` — note `customer_id` is not a partition column, 
so only the state-level pruning applies here (down to `customer_state=PR/`, then scan across all 
city sub-folders within it for the matching `customer_id`).

In [25]:
# Filter on customer_state (partition column, prunes to state=PR/) + customer_id (not partitioned,
# scanned within that pruned subset)
spark.sql("select * from customers where customer_state = 'PR' and customer_id = 19").show()

+-----------+--------------+--------------+--------------+-----------------+--------------------+----------------+--------------+-------------+
|customer_id|customer_fname|customer_lname|customer_email|customer_password|     customer_street|customer_zipcode|customer_state|customer_city|
+-----------+--------------+--------------+--------------+-----------------+--------------------+----------------+--------------+-------------+
|         19|     Stephanie|      Mitchell|     XXXXXXXXX|        XXXXXXXXX|3543 Red Treasure...|             725|            PR|       Caguas|
+-----------+--------------+--------------+--------------+-----------------+--------------------+----------------+--------------+-------------+



### Query using BOTH partition levels (state + city)
This is the most efficient case: filtering on both `customer_state` AND `customer_city` lets 
Spark prune all the way down to the exact leaf folder `customer_state=PR/customer_city=Caguas/`.

In [26]:
# Best case for pruning: filters on BOTH partition columns -> Spark reads only the exact
# customer_state=PR/customer_city=Caguas/ folder
spark.sql("select count(*) from customers where customer_state = 'PR' and customer_city = 'Caguas'").show()

+--------+
|count(1)|
+--------+
|    4584|
+--------+



### Query using only the SECOND-level partition column (city, without state)
Here we filter only on `customer_city`, skipping `customer_state` (the higher-level partition). 
Since `customer_city` sub-folders exist under every `customer_state=.../` folder, Spark generally 
still needs to look inside every state folder to find matching city folders — pruning is 
**less effective** than when the top-level partition column is also filtered. 
Same result (4584) as the query above, but the physical scan behind it is less optimal — 
this shows why the **order of columns in `partitionBy()`** matters for how well common filter 
patterns get pruned.

In [27]:
# Filtering only on the second-level partition column (customer_city), skipping the top-level
# customer_state filter -> Spark must look under every state folder for a customer_city=Caguas match
spark.sql("select count(*) from customers where customer_city = 'Caguas'").show()

+--------+
|count(1)|
+--------+
|    4584|
+--------+

